# Business Entity Resolution — run on Colab

CPU-only pipeline (no GPU needed). Use **Runtime → Change runtime type → High-RAM** (Colab Pro).

Intermediate files live on Google Drive, so if the runtime disconnects you can reconnect, re-run cells 2–4 and resume from the step that died (cell 7).

Before starting: upload `6ab10eb3b23ba_student_resource.zip` to the root of **My Drive** (or change `ZIP` below).

In [ ]:
# 1. What machine did we get? Full run wants >= ~40 GB RAM; with --train-pct 30 ~25 GB.
!nproc; free -g; df -h /content | tail -1

In [ ]:
# 2. Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

import os
ZIP = '/content/drive/MyDrive/6ab10eb3b23ba_student_resource.zip'
os.environ['ER_WORK_DIR'] = '/content/drive/MyDrive/er_work'      # persists across disconnects
os.environ['ER_OUTPUT_DIR'] = '/content/drive/MyDrive/er_output'
os.environ['ER_BLOCK_CHUNK'] = '100000'                             # lower peak memory
os.environ['PYTHONIOENCODING'] = 'utf-8'

In [ ]:
# 3. Code + data (data is unzipped to fast local disk, not Drive)
!test -d /content/er || git clone https://github.com/vitthalg17/amazon-ml-challenge-2026-entity-resolution.git /content/er
!cd /content/er && git pull -q
!test -d /content/er/student_resource || (cd /content/er && unzip -q "$ZIP" -x "__MACOSX/*")
!ls /content/er/student_resource/dataset/test

In [ ]:
# 4. Dependencies
!pip install -q -r /content/er/business_entity_resolution/requirements-min.txt

In [ ]:
# 5. Smoke test (~15 min): 2% of Source 2/3, ends with the official validator -> PASS
#    Uses separate folders so it never mixes with the real run.
!cd /content/er/business_entity_resolution/src && \
  ER_WORK_DIR=/content/smoke_work ER_OUTPUT_DIR=/content/smoke_out python -u pipeline.py --pct 2 2>&1 | grep -E '=====|Traceback|Error|best threshold|PASS|FAIL'

In [ ]:
# 6. Real run in the background: train on 30% of train Source 2/3, predict on 100% of test.
#    (Drop --train-pct 30 if cell 1 showed >= ~50 GB RAM.)
!cd /content/er/business_entity_resolution/src && \
  nohup python -u pipeline.py --train-pct 30 > /content/drive/MyDrive/er_run.log 2>&1 &
print('started; watch progress with the next cell')

In [ ]:
# 7. Progress (re-run this cell whenever you want). Resume after a disconnect with e.g.:
#    !cd /content/er/business_entity_resolution/src && nohup python -u pipeline.py --train-pct 30 \
#        --steps block_test match_features_train match_features_test train predict validate \
#        >> /content/drive/MyDrive/er_run.log 2>&1 &
!grep -E '=====|raw ->|Traceback|Error|MemoryError|best threshold|PASS|FAIL' /content/drive/MyDrive/er_run.log | tail -25
!free -g | head -2

In [ ]:
# 8. When the log shows PASS: outputs are in Drive -> er_output/
#    matching_results.tsv  -> upload to the leaderboard
#    candidate_pairs.tsv   -> keep for the final zip
!ls -la /content/drive/MyDrive/er_output/